In [21]:
from pipeline_utils import load_and_combine_data, clean_data, create_features, handle_cardinality, prepare_ml_dataframe
import pandas as pd
from xgboost import XGBClassifier
from pathlib import Path
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    precision_score, 
    recall_score, 
    f1_score
)

PREDICTOR_FEATURES = ['item_price', 'is_boosted', 'hashtag_count', 'description_word_count', 'price_ratio', 'buyer_shipping_cost']
CATEGORICAL_TARGETS = ['brand', 'category', 'day_posted', 'season']
TARGET_VARIABLE = 'fast_sale'

In [20]:
# Executing the data pipeline

df = load_and_combine_data()
df = clean_data(df)
df = create_features(df)
df = handle_cardinality(df,'brand')

# Prepare the ML dataframe

ml_df, final_feature_list = prepare_ml_dataframe(
    df,
    PREDICTOR_FEATURES,
    CATEGORICAL_TARGETS,
    [TARGET_VARIABLE]
)


print(df['fast_sale'].value_counts())


Found 4 files: ['dec_sales.csv', 'nov_sales.csv', 'oct_sales.csv', 'sep_sales.csv']
Successfully loaded dec_sales.csv with 99 rows.
Successfully loaded nov_sales.csv with 101 rows.
Successfully loaded oct_sales.csv with 57 rows.
Successfully loaded sep_sales.csv with 44 rows.

Total rows in combined dataset: 301
fast_sale
False    166
True     135
Name: count, dtype: int64


In [19]:
X = ml_df[final_feature_list]
y = ml_df[TARGET_VARIABLE]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train
clf = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
clf.fit(X_train, y_train)

# Get Predictions
y_pred = clf.predict(X_test)
y_probs = clf.predict_proba(X_test)[:, 1] 
threshold = 0.5
y_pred_custom = (y_probs >= threshold).astype(int)

# 4. Detailed Metrics
print(f"--- Model Performance Summary ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_custom):.2%}")
print(f"Precision: {precision_score(y_test, y_pred_custom):.2%}")
print(f"Recall:    {recall_score(y_test, y_pred_custom):.2%}")
print(f"F1-Score:  {f1_score(y_test, y_pred_custom):.2%}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_probs):.2%}")

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred_custom))

print("\n--- Full Classification Report ---")
print(classification_report(y_test, y_pred_custom, target_names=['Slow Sale', 'Fast Sale']))

--- Model Performance Summary ---
Accuracy:  63.93%
Precision: 55.17%
Recall:    64.00%
F1-Score:  59.26%
ROC-AUC:   66.22%

--- Confusion Matrix ---
[[23 13]
 [ 9 16]]

--- Full Classification Report ---
              precision    recall  f1-score   support

   Slow Sale       0.72      0.64      0.68        36
   Fast Sale       0.55      0.64      0.59        25

    accuracy                           0.64        61
   macro avg       0.64      0.64      0.63        61
weighted avg       0.65      0.64      0.64        61

